# Grouped-Query Attention — Solution

Reference implementation for `02_gqa_exercise.ipynb`.

## Setup

In [2]:
import torch
import torch.nn as nn

## Solution 1 — `GroupedQueryAttention`

In [ ]:
# 8 heads
# 64 - d_model

# q1,q2,q3,q4,q5,q6,q7,q8
# 8,8,8,8,8,8,8,8

# q -> 8,seq_len,64

#nkv_heads = 4 
# key vector - 8*4
# k -> 4,seq_len,64
# k1,k2,k3,k4 , two query vector share one key vector 
# value vector - 8*4
# v1,v2,v3,v4 , two query vector share one value vector

In [7]:
(torch.randn(2,8,1,16,2) @ torch.randn(2,1,4,2,16)).shape

torch.Size([2, 8, 4, 16, 16])

In [28]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, causal=False, max_seq_len=4096):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.causal = causal
        self.max_seq_len = max_seq_len
        self.d_head = self.d_model//self.n_heads
        self.groups = self.n_heads//self.n_kv_heads

        self.qkv_projection = nn.Linear(self.d_model,self.d_model+2*self.d_head*self.n_kv_heads)
        self.out_projections = nn.Linear(self.d_model,self.d_model)

        self.register_buffer("causal_mask",torch.triu(torch.ones(max_seq_len,max_seq_len,dtype=torch.bool),diagonal=1))

    def forward(self, x):
        bs,seq_len,_ = x.shape

        Q,K,V = torch.split(self.qkv_projection(x),[self.d_model,self.d_head*self.n_kv_heads,self.d_head*self.n_kv_heads],dim=-1)
        Q = Q.view(bs,seq_len,self.n_heads,self.d_head).permute(0,2,1,3) #bs,n-heads,seq,d-model
        K = K.view(bs,seq_len,self.n_kv_heads,self.d_head).permute(0,2,1,3) #bs,n-kv_heads,seq,d-model
        V = V.view(bs,seq_len,self.n_kv_heads,self.d_head).permute(0,2,1,3) #bs,n-kv_heads,seq,d-model

        ## Need to replicate K and V to match n_heads
        #K = K.repeat_interleave(self.groups,dim=1)
        K = K[:,:,None,:,:].expand(bs,self.n_kv_heads,self.groups,seq_len,self.d_head).reshape(bs,self.n_kv_heads*self.groups,seq_len,self.d_head)
        V = V[:,:,None,:,:].expand(bs,self.n_kv_heads,self.groups,seq_len,self.d_head).reshape(bs,self.n_kv_heads*self.groups,seq_len,self.d_head)
        
        scores = Q@K.transpose(-2,-1) / self.d_head**0.5

        if self.causal:
            scores = scores.masked_fill(self.causal_mask[:seq_len,:seq_len],float("-inf"))

        weights = torch.softmax(scores,dim=-1)
        out = weights@V
        out = out.permute(0,2,1,3).contiguous().view(bs,seq_len,self.d_model)

        out = self.out_projections(out)

        return out

**Why `repeat_interleave` and not `repeat`?** _Explain: `repeat_interleave(N, dim=1)` produces `[h0, h0, h0, h0, h1, h1, h1, h1, ...]` — keeps groupings adjacent. `repeat(N, ...)` would produce `[h0, h1, h0, h1, ...]` — interleaved instead of grouped. Wrong grouping = wrong shared assignment._

**Why GQA won.** _GQA captures most of MQA's memory savings (e.g., 8× from `n_kv_heads=8` to `1` is unnecessary; `n_kv_heads=8` already gives 4× over MHA at `n_heads=32`) while preserving most of the per-head diversity that pure MQA sacrifices. Sweet spot of the trade-off curve._

## Solution 2 — Sweep cache sizes

In [29]:
d_model, n_heads = 512, 8
d_head = d_model // n_heads
seq_len = 8192    # representative inference context

print(f"d_model = {d_model}, n_heads = {n_heads}, d_head = {d_head}")
print()
print(f"{'n_kv_heads':>10}  {'mode':>4}  {'W_k weights':>12}  {'reduction vs MHA':>17}  {'KV cache @8K (MB)':>18}")
print("-" * 78)

ref_wk = d_model * d_model    # full MHA W_k size (d_model × d_model)

for n_kv in [1, 2, 4, 8]:
    # W_k portion of the fused QKV projection: d_model -> n_kv * d_head
    wk_weights = d_model * (n_kv * d_head)
    reduction  = ref_wk / wk_weights

    # Per-layer KV cache: 2 (K+V) × n_kv_heads × seq_len × d_head × 2 bytes (fp16)
    cache_bytes = 2 * n_kv * seq_len * d_head * 2

    mode = "MQA" if n_kv == 1 else "MHA" if n_kv == n_heads else "GQA"
    print(f"{n_kv:>10}  {mode:>4}  {wk_weights:>12,}  {reduction:>15.1f}x  {cache_bytes / 1e6:>18.1f}")

d_model = 512, n_heads = 8, d_head = 64

n_kv_heads  mode   W_k weights   reduction vs MHA   KV cache @8K (MB)
------------------------------------------------------------------------------
         1   MQA        32,768              8.0x                 2.1
         2   GQA        65,536              4.0x                 4.2
         4   GQA       131,072              2.0x                 8.4
         8   MHA       262,144              1.0x                16.8


## Run the tests

In [30]:
from tests import run_gqa_tests
run_gqa_tests(GroupedQueryAttention)

Running GroupedQueryAttention...
  ✓ Output shape correct
  ✗ Asserts n_heads divisible by n_kv_heads — n_heads % n_kv_heads != 0 should raise
  ✓ W_k size scales with n_kv_heads
  ✓ Causal mode works

  3/4 checks passed — fix the ✗ above



False